# House Price Prediction - Internship Project Week 1

**Student:** Kartik Chadda  
**Goal:** Build regression models to predict house prices, compare model accuracy, visualize key patterns, and write insights.

## Task 1 - Data Loading and Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
df = pd.read_csv("Housing.csv")
df.head(10)

In [ ]:
print("Rows and columns:", df.shape)
print("Columns:", df.columns.tolist())

target = "price"
features = [col for col in df.columns if col != target]
print("Target column:", target)
print("Feature columns:", features)

df.info()

In [ ]:
df.isnull().sum()

## Task 2 - Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Fill missing values if any appear in future versions of the dataset
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Keep meaningful columns only
meaningful_columns = [
    "price", "area", "bedrooms", "bathrooms", "stories", "mainroad",
    "guestroom", "basement", "hotwaterheating", "airconditioning",
    "parking", "prefarea", "furnishingstatus"
]
df = df[meaningful_columns]

# One-hot encoded preview for correlation/model understanding
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

## Task 3 - Model Building

In [ ]:
X = df.drop("price", axis=1)
y = df["price"]

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

models = {
    "Linear Regression": Pipeline([
        ("preprocess", preprocess),
        ("model", LinearRegression())
    ]),
    "Random Forest Regressor": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestRegressor(n_estimators=300, random_state=42, max_depth=10))
    ])
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
        "R2 Score": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
best_model_name = results_df.sort_values("R2 Score", ascending=False).iloc[0]["Model"]
best_model_name

## Task 4 - Visualization

In [ ]:
plt.figure(figsize=(9, 6))
plt.hist(df["price"], bins=30)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("charts/price_distribution.png", dpi=200)
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
corr = df_encoded.corr(numeric_only=True)
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=6)
plt.yticks(range(len(corr.index)), corr.index, fontsize=6)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("charts/correlation_heatmap.png", dpi=200)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions[best_model_name], alpha=0.75)
min_value = min(y_test.min(), predictions[best_model_name].min())
max_value = max(y_test.max(), predictions[best_model_name].max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title(f"Actual vs Predicted House Price - {best_model_name}")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.tight_layout()
plt.savefig("charts/actual_vs_predicted.png", dpi=200)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for label, group in df.groupby("airconditioning"):
    plt.scatter(group["area"], group["price"], alpha=0.65, label=f"airconditioning={label}")
plt.legend()
plt.title("Price vs Area by Air Conditioning")
plt.xlabel("Area")
plt.ylabel("Price")
plt.tight_layout()
plt.savefig("charts/price_vs_area.png", dpi=200)
plt.show()

## Feature Importance

In [ ]:
rf_model = models["Random Forest Regressor"]
feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
top_fi = feature_importance.head(10).sort_values("importance")
plt.barh(top_fi["feature"], top_fi["importance"])
plt.title("Top 10 Most Important Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("charts/top_feature_importance.png", dpi=200)
plt.show()

## Task 5 - Insights and Summary

The most influential features for house price prediction are **area**, **bathrooms**, **air conditioning**, **preferred area**, and **parking availability**. The model results show that **Linear Regression** performed best on the test set, with an R2 score of approximately **0.91**, which means it explains a strong part of the price variation in plain terms. A surprising pattern is that comfort/location features like air conditioning and preferred area can increase price significantly, not only the property size. The price distribution is right-skewed, meaning most houses are in the middle price range and only a few are very expensive. A real estate business should focus its pricing strategy on area, bathrooms, furnishing status, and premium facilities because these features strongly affect market value.